# A01 — Portfolio Optimiser (Manager Edition)

> One clear question: **Which features should we fund now?**

This notebook uses **`advanced-features.yaml`** and gives one decision flow:
1. Rank features by value and downside floor
2. Compare Exact, ILP, and Greedy optimisation
3. Check concentration risk before final funding

## Step 0 — Before You Start (Quick Glossary)

| Term | Simple meaning |
|---|---|
| VaR 95% (Floor) | In 95 out of 100 simulations, business value stays above this value |
| LLP | Probability that a feature is delayed or not delivered |
| Exact solver | Tests every possible feature combination |
| ILP solver | Fast mathematical shortcut for portfolio selection |
| HHI | Concentration score: high means too much business value depends on few features |

📌 **Key takeaway**: Select one optimised portfolio, then confirm floor and concentration before funding.

## Step 1 — Load `advanced-features.yaml` and run one simulation baseline

In [ ]:
from fhs.application import AdvancedPortfolioService
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import COLORS, show
from fhs.presentation.notebook.charts import (
    plot_business_value_concentration,
    plot_feature_ranking,
    plot_llp_landscape,
    plot_strategy_category_mix,
)

setup = notebook_setup("advanced-features")

In [ ]:
scenario = setup.scenario

if scenario is None:
    raise RuntimeError("Scenario setup could not be initialized")

features = scenario.features
service = AdvancedPortfolioService.from_scenario(
    scenario,
    seed=scenario.seed,
    scenarios=scenario.scenarios,
)
_ = service.simulation_results

show.info(
    f"Source: <code>{scenario.config_path}</code><br>"
    f"Features: <b>{len(features)}</b> · Budget: <b>EUR {scenario.budget:,.0f}</b>"
)

## Step 2 — Rank features by value, floor, and delivery exposure

In [ ]:
show.section("Feature Ranking — Value View")

ranking_rows = service.decisions.rank_features()

plot_feature_ranking(
    [
        {
            "feature": row.feature,
            "expected_business_value": row.expected_business_value,
            "var_95_business_value": row.var_95_business_value,
            "risk_ratio": row.risk_ratio,
        }
        for row in ranking_rows
    ],
    title="Feature Priority — Expected vs Floor (VaR 95%)",
);

### Delivery Risk per Feature

The chart and table below add the delivery risk dimension (LLP = probability of late or failed delivery).

In [ ]:
show.section("Feature Ranking — Delivery Risk View")

plot_llp_landscape(
    features,
    service.simulation_results,
    title="Feature Map — Delivery Risk (LLP) vs Expected Business Value",
);

In [ ]:
show.feature_ranking_table(
    [
        (
            str(idx),
            row.feature,
            f"EUR {row.cost:,.0f}",
            f"EUR {row.expected_business_value:,.0f}",
            f"EUR {row.var_95_business_value:,.0f}",
            f"{row.llp:.0%}",
        )
        for idx, row in enumerate(ranking_rows, 1)
    ],
    title="Feature Ranking Table",
)

## Step 3 — Compare solvers at current budget (Exact vs ILP vs Greedy)

**How the methods differ (simple):**

- **Exact** checks every valid feature combination under budget. It is the quality benchmark.
- **ILP** uses a fast mathematical optimisation model. Usually much faster, often same answer.
- **Greedy** adds features one by one by score. Fastest, but can miss the best combination.

For `var_floor`, keep this in mind:
- Exact optimises the real portfolio floor from combined scenarios.
- ILP optimises a linear proxy from individual feature scores.
- Because of this, ILP and Exact can differ in edge cases (especially with negative individual scores).

In [ ]:
show.section("Solver Comparison")

solver_results = service.compare_solvers(
    budget=scenario.budget,
    strategy="var_floor",
    include=("exact", "ilp", "greedy"),
)

In [ ]:
show.solver_compare(
    solver_results["exact"],
    solver_results["ilp"],
    solver_results["greedy"],
    title="Portfolio Optimization — Exact vs Linear (ILP) vs Greedy",
)

In [ ]:
rows = service.decisions.solver_comparison_rows(
    solver_results,
    include=("exact", "ilp", "greedy"),
)

show.solver_results(rows)

## Step 4 — Validate concentration (HHI) and strategy balance

In [ ]:
show.section("Portfolio Concentration")

chosen = solver_results["ilp"]
selected_names = chosen["recommended_features"]

concentration = service.decisions.concentration(selected_names)
shares_sorted = sorted(
    concentration.shares.items(), key=lambda item: item[1], reverse=True
)

plot_business_value_concentration(
    {
        "concentration": dict(shares_sorted),
        "hhi": concentration.hhi,
        "is_concentrated": concentration.hhi >= 0.25,
    },
    title_prefix="Hirschman Concentration — Selected Portfolio",
);

### Strategy Balance & Final Decision

The donut chart shows how investment is spread across strategy categories. Below it, the decision metrics and portfolio details confirm the GO / Review signal.

In [ ]:
show.section("Strategy Balance & Decision")

category_rows = service.decisions.strategy_category_cost_share(
    selected_names,
    scenario.strategy,
)
plot_strategy_category_mix(
    [
        {"category": row.category, "cost": row.cost, "share": row.share}
        for row in category_rows
    ],
    title="Selected Portfolio Mix by Strategy Category (Cost Share)",
);

In [ ]:
hhi = concentration.hhi
show.metrics(
    [
        ("Selected method", "Linear (ILP)", None),
        ("Selected features", str(len(selected_names)), None),
        (
            "Portfolio floor (VaR 95%)",
            f"EUR {chosen['portfolio_var_95']:,.0f}",
            COLORS.danger,
        ),
        (
            "Safety buffer (Floor - Cost)",
            f"EUR {chosen['safety_buffer']:,.0f}",
            COLORS.secondary if chosen["safety_buffer"] >= 0 else COLORS.warning,
        ),
        (
            "HHI (Hirschman)",
            f"{hhi:.2f}",
            COLORS.warning if hhi >= 0.25 else COLORS.secondary,
        ),
    ],
    title="Final Decision Snapshot",
)

In [ ]:
rows = service.decisions.selected_portfolio_rows(
    selected_names,
    scenario.strategy,
)

show.selected_portfolio_details(rows)

if chosen["safety_buffer"] >= 0 and hhi < 0.25:
    show.success("GO signal: floor is above cost and concentration is acceptable.")
elif chosen["safety_buffer"] < 0:
    show.warning("Review signal: floor is below cost. Re-check budget or feature mix.")
else:
    show.warning(
        "Review signal: concentration is high. Add one independent feature if possible."
    )

## Summary

Manager decision flow completed:
1. Prioritize features by value and floor.
2. Compare solver methods under one budget (Exact, ILP, Greedy).
3. Validate concentration risk (HHI) before funding.

## Next Notebook
- Portfolio investment decisions at scale: **A02 Portfolio Investment Decision**.

## Notebook Glossary (A01)
- **Business Value Floor 95**: Conservative planning floor. In 95 of 100 cases, business value stays above this value.
- **Safety Buffer**: `Floor - Investment Cost`. Positive means downside floor still covers spend.
- **Linear (ILP)**: Integer Linear Programming; practical default for budget-constrained portfolio selection.
- **Exact Solver**: Benchmark method for quality; can become slow as feature count grows.
- **Greedy Solver**: Fast approximation; useful when runtime matters more than perfect quality.
- **HHI (Hirschman)**: Concentration score from business value shares. Higher means more dependency on few features.